In [21]:
!ls /kaggle/input/safetyeye/data

test  train  valid


In [22]:
%%writefile data.yaml
train: /kaggle/input/safetyeye/data/train/images
val: /kaggle/input/safetyeye/data/valid/images
nc: 7
names: ['person','helmet','no_helmet','vest','no_vest','machinery','cone']



Overwriting data.yaml


In [26]:
import os
import shutil
from PIL import Image

# -------------------------------
# 1️⃣ Paths
# -------------------------------
src_images = "/kaggle/input/safetyeye/data/train/images"
src_labels = "/kaggle/input/safetyeye/data/train/labels"

dst_base = "/kaggle/working/clean_dataset"
dst_images = os.path.join(dst_base, "images")
dst_labels = os.path.join(dst_base, "labels")
os.makedirs(dst_images, exist_ok=True)
os.makedirs(dst_labels, exist_ok=True)

# -------------------------------
# 2️⃣ Class remapping
# Old IDs -> New IDs (0-6)
# -------------------------------
# Example: your dataset may have old IDs 6, 9 etc.
# Map them to correct new class IDs
remap = {
    0: 0,  # person
    1: 1,  # helmet
    2: 2,  # no_helmet
    3: 3,  # vest
    4: 4,  # no_vest
    5: 5,  # machinery
    6: 6,  # cone
    9: 1   # Example: remap old class 9 to helmet
}

# -------------------------------
# 3️⃣ Clean and copy
# -------------------------------
removed_count = 0
fixed_count = 0
duplicates = set()

for img_file in os.listdir(src_images):
    if not img_file.lower().endswith((".jpg",".png")):
        continue
    
    img_path = os.path.join(src_images, img_file)
    label_file = img_file.rsplit(".",1)[0]+".txt"
    label_path = os.path.join(src_labels, label_file)
    
    # Check if image is valid
    try:
        img = Image.open(img_path)
        img.verify()
    except:
        removed_count += 1
        continue
    
    # Skip duplicates
    if img_file in duplicates:
        removed_count += 1
        continue
    duplicates.add(img_file)
    
    # Process labels
    if os.path.exists(label_path):
        new_lines = []
        with open(label_path, "r") as f:
            for line in f.readlines():
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                old_id = int(parts[0])
                if old_id in remap:
                    parts[0] = str(remap[old_id])
                    fixed_count += 1
                    new_lines.append(" ".join(parts))
                else:
                    # Ignore labels with invalid class IDs
                    removed_count += 1
        
        # Write new label file if lines exist
        if new_lines:
            with open(os.path.join(dst_labels, label_file), "w") as f:
                f.write("\n".join(new_lines))
        else:
            removed_count += 1
            continue
    else:
        removed_count += 1
        continue
    
    # Copy image
    shutil.copy(img_path, dst_images)

print(f"✅ Finished cleaning dataset: {fixed_count} labels fixed, {removed_count} images/labels removed.")
print(f"Clean images saved at: {dst_images}")
print(f"Clean labels saved at: {dst_labels}")


✅ Finished cleaning dataset: 26468 labels fixed, 0 images/labels removed.
Clean images saved at: /kaggle/working/clean_dataset/images
Clean labels saved at: /kaggle/working/clean_dataset/labels


In [29]:
data_yaml_path = "/kaggle/working/data.yaml"

with open(data_yaml_path, "w") as f:
    f.write("""
train: /kaggle/working/clean_dataset/images
val: /kaggle/working/clean_dataset/images  # replace with validation folder if available
test: /kaggle/working/clean_dataset/images # replace with test folder if available
nc: 7
names: ['person','helmet','no_helmet','vest','no_vest','machinery','cone']
""")

print(f"✅ data.yaml saved at {data_yaml_path}")


✅ data.yaml saved at /kaggle/working/data.yaml


 Create data.yaml

In [30]:
from ultralytics import YOLO

# Load a bigger model (small version for higher accuracy than yolov8n)
model = YOLO("yolov8s.pt")

model.train(
    data="data.yaml",        
    epochs=30,               
    imgsz=768,               
    batch=16,                
    device=0,                
    optimizer="SGD",         
    lr0=0.002,               
    lrf=0.01,                
    momentum=0.937,          
    weight_decay=0.0005,     
    patience=20,             
    name="safetyeye_final"
)

Ultralytics 8.3.203 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=768, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.002, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=safetyeye_final, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience=20, perspective=0.0, plots=True, pose=12.0, pretrain

/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all       1870      26466        0.9       0.72      0.809      0.582
                person       1820       9042      0.924      0.795      0.876      0.675
                helmet       1149       2334      0.903      0.805      0.879      0.677
             no_helmet       1480       3840      0.899      0.712      0.795      0.555
                  vest       1839       8832      0.905      0.766       0.85      0.651
                  cone        448       2418       0.87      0.523      0.646      0.355
Speed: 0.3ms preprocess, 7.0ms inference, 0.0ms loss, 1.5ms postprocess per image
Results saved to /kaggle/working/runs/detect/safetyeye_final


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 6])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7ba5be674d10>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
        

📊 YOLOv8 Validation Results & Predictions

Below are the confusion matrix, precision/recall curves, and sample predicted images with bounding boxes.
